In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10,6)

PATH = "superstore_dataset.csv"

df = pd.read_csv(PATH)

print("Dataset shape:", df.shape)
display(df.head())
df.info()


In [ ]:

print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after:", df.duplicated().sum())

#  missing values
print("\nMissing values per column:\n", df.isnull().sum())

if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)

for col in ['Order Date', 'Ship Date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

if 'Order Date' in df.columns:
    df['Order Year'] = df['Order Date'].dt.year
    df['Order Month'] = df['Order Date'].dt.month
    df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

df['Profit Margin %'] = (df['Profit'] / df['Sales']).replace([np.inf, -np.inf], np.nan) * 100

display(df[['Sales','Profit','Profit Margin %']].describe().round(2))


In [ ]:
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=False)
display(state_sales.head(20))

top_states = state_sales.head(20).sort_values()
plt.figure(figsize=(10,8))
sns.barplot(x=top_states.values, y=top_states.index, palette='viridis')
plt.title("Top 20 States by Total Sales")
plt.xlabel("Total Sales ($)")
plt.ylabel("State")
plt.tight_layout()
plt.show()


In [ ]:
# Compare NY vs CA
for st in ['New York', 'California']:
    if st in df['State'].values:
        total_sales = df[df['State']==st]['Sales'].sum()
        total_profit = df[df['State']==st]['Profit'].sum()
        print(f"{st} -> Sales: ${total_sales:,.0f} | Profit: ${total_profit:,.0f}")
    else:
        print(f"{st} not in dataset")

states_comp = df[df['State'].isin(['New York','California'])].groupby('State')[['Sales','Profit']].sum()
states_comp.plot(kind='bar', rot=0)
plt.title("Sales and Profit: New York vs California")
plt.ylabel("Amount ($)")
plt.show()


In [ ]:
state_summary = df.groupby('State').agg({'Sales':'sum','Profit':'sum'}).assign(**{'Profit Margin %': lambda x: x['Profit']/x['Sales']*100})
state_summary = state_summary.sort_values('Profit Margin %', ascending=False)
display(state_summary.head(20).round(2))

# Plot Profit Margin for top 20 revenue states
top20_rev = state_sales.head(20).index
plot_df = state_summary.loc[top20_rev].sort_values('Profit Margin %')
plt.figure(figsize=(10,8))
sns.barplot(x='Profit Margin %', y=plot_df.index, data=plot_df, palette='coolwarm')
plt.xlabel("Profit Margin (%)")
plt.title("Profit Margin % for Top 20 Revenue States")
plt.tight_layout()
plt.show()


In [ ]:
# Pareto: customers contributing to profit

cust_profit = df.groupby('Customer ID')['Profit'].sum().sort_values(ascending=False).reset_index()
cust_profit['cum_profit'] = cust_profit['Profit'].cumsum()
total_profit = cust_profit['Profit'].sum()
cust_profit['cum_profit_pct'] = cust_profit['cum_profit'] / total_profit

#  customer counts and percent of customers
cust_profit['customer_rank'] = np.arange(1, len(cust_profit)+1)
cust_profit['cust_pct'] = cust_profit['customer_rank'] / len(cust_profit)

# Find percent of profit from top 20% customers
top_20_cut = int(0.2 * len(cust_profit))
profit_from_top20pct = cust_profit.loc[:top_20_cut-1, 'Profit'].sum()
print(f"Top 20% customers (count={top_20_cut}) contribute ${profit_from_top20pct:,.0f} which is {profit_from_top20pct/total_profit*100:.1f}% of total profit")

# Plot cumulative curve
plt.figure(figsize=(10,6))
plt.plot(cust_profit['cust_pct']*100, cust_profit['cum_profit_pct']*100, marker='o')
plt.axvline(20, color='red', linestyle='--', label='20% customers')
plt.axhline(80, color='green', linestyle='--', label='80% profit')
plt.xlabel('Percent of Customers (%)')
plt.ylabel('Cumulative Percent of Profit (%)')
plt.title('Pareto Curve: Customers vs Cumulative Profit')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Top 20 cities by Sales and Profit
city_sales = df.groupby('City')['Sales'].sum().sort_values(ascending=False)
city_profit = df.groupby('City')['Profit'].sum().sort_values(ascending=False)

top20_cities_sales = city_sales.head(20)
top20_cities_profit = city_profit.head(20)

print("Top 20 Cities by Sales:")
display(top20_cities_sales)

print("\nTop 20 Cities by Profit:")
display(top20_cities_profit)

# Compare overlap
overlap = set(top20_cities_sales.index).intersection(set(top20_cities_profit.index))
print(f"\nOverlap between top-20 sales and top-20 profit cities: {len(overlap)} cities")
print(sorted(list(overlap)))


In [ ]:
#  Top 20 customers by Sales
cust_sales = df.groupby(['Customer ID','Customer Name'])['Sales'].sum().sort_values(ascending=False).reset_index()
display(cust_sales.head(20))

# Cumulative curve for sales by customers
cust_sales['cum_sales'] = cust_sales['Sales'].cumsum()
total_sales = cust_sales['Sales'].sum()
cust_sales['cum_sales_pct'] = cust_sales['cum_sales'] / total_sales
cust_sales['cust_pct'] = (np.arange(1, len(cust_sales)+1) / len(cust_sales)) * 100

plt.figure(figsize=(10,6))
plt.plot(cust_sales['cust_pct'], cust_sales['cum_sales_pct']*100, marker='o')
plt.axvline(20, color='red', linestyle='--', label='20% customers')
plt.axhline(80, color='green', linestyle='--', label='80% sales')
plt.xlabel('Percent of Customers (%)')
plt.ylabel('Cumulative Percent of Sales (%)')
plt.title('Pareto Curve: Customers vs Cumulative Sales')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

top20cust_count = int(0.2 * len(cust_sales))
sales_from_top20 = cust_sales.loc[:top20cust_count-1, 'Sales'].sum()
print(f"Top 20% customers contribute {sales_from_top20/total_sales*100:.1f}% of total sales")


In [ ]:
# Strategic recommendations based on findings

# Top states to prioritize by sales
priority_states = state_sales.head(10)
print("States to prioritize by Sales (top 10):")
display(priority_states)

# Top cities to prioritize by profit and sales overlap
priority_cities = list(set(top20_cities_sales.head(10).index).union(set(top20_cities_profit.head(10).index)))
print("Cities to prioritize (union of top sales/profit):", priority_cities[:20])

print("\nRecommendations:")
print("1. Focus marketing & promotions in the top revenue states (shown above).")
print("2. For New York and California: CA may have higher/lower profit margin — run category-level discount tests before scaling discounts.")
print("3. Implement VIP/retention programs for top 20% customers (they drive a large share of sales/profit).")
print("4. Investigate cities in the top-20 lists with low profit margins for cost/repricing optimization.")
print("5. Limit blanket high discounts (>20%) where we observe frequent losses; test controlled promotions instead.")


In [ ]:
state_sales.head(50).to_csv("state_sales_summary.csv")
state_summary.head(50).to_csv("state_profit_margin_summary.csv")
top20_cities_sales.to_csv("top20_cities_sales.csv")
top20_cities_profit.to_csv("top20_cities_profit.csv")
cust_sales.head(200).to_csv("customer_sales_ranking.csv")
cust_profit.head(200).to_csv("customer_profit_ranking.csv")

